# FAOSTAT Olive Production - Exploration and Cleaning

**Module Project 1: Data Storytelling**  
**Alberto Gomez Soteres & Burak Donbekci**

This notebook explores and cleans FAOSTAT olive data for Spain and Türkiye.

We focus on three main agricultural measures: production, harvested area, and yield. The goal is to understand the dataset, check its quality, and prepare clean data for the next stages of the project.

## 1. Imports

First, we import the tools needed to load, explore, and clean the data.

In [1]:
from pathlib import Path

import pandas as pd

## 2. Load the Data

Next, we load the raw FAOSTAT dataset from the project's data folder.

In [2]:
# Define the project root
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Load the raw FAOSTAT dataset
file_path = project_root / "data" / "raw" / "faostat_olives.csv"

df = pd.read_csv(file_path)

# Look at the first rows
df.head()

,faostat,m49 code,Country,Item Code,Item,Year,Yield (kg/ha),Yield (kg/ha) flag,Production (tonnes),Production (tonnes) flag,Area harvested (ha),Area harvested (ha) flag
0,68,250,France,260,Olives,1961,44.9,A,2009.0,A,44756.0,A
1,124,434,Libya,260,Olives,1961,NaN,NaN,35100.0,A,NaN,M
2,138,484,Mexico,260,Olives,1961,1419.6,A,3691.0,A,2600.0,A
3,231,840,United States of America,260,Olives,1961,3498.2,A,39915.0,A,11410.0,A
4,248,890,Yugoslav SFR,260,Olives,1961,NaN,NaN,28300.0,A,NaN,M


## 3. Understand the Dataset Structure

Before cleaning the data, we check its size, columns, and data types. This helps us understand how the FAOSTAT dataset is organized.

In [3]:
# First, we check the size of the dataset
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 2481
Columns: 12


In [4]:
# Now, we check the available columns
df.columns.tolist()

['faostat',
 'm49 code',
 'Country',
 'Item Code',
 'Item',
 'Year',
 'Yield (kg/ha)',
 'Yield (kg/ha) flag',
 'Production (tonnes)',
 'Production (tonnes) flag',
 'Area harvested (ha)',
 'Area harvested (ha) flag']

In [5]:
# Finally, we check the data types and non-null values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2481 entries, 0 to 2480
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   faostat                   2481 non-null   int64  
 1   m49 code                  2481 non-null   int64  
 2   Country                   2481 non-null   str    
 3   Item Code                 2481 non-null   int64  
 4   Item                      2481 non-null   str    
 5   Year                      2481 non-null   int64  
 6   Yield (kg/ha)             2067 non-null   float64
 7   Yield (kg/ha) flag        2067 non-null   str    
 8   Production (tonnes)       2313 non-null   float64
 9   Production (tonnes) flag  2480 non-null   str    
 10  Area harvested (ha)       2062 non-null   float64
 11  Area harvested (ha) flag  2478 non-null   str    
dtypes: float64(3), int64(4), str(5)
memory usage: 232.7 KB


## 4. Dataset Coverage

Next, we check the scope of the dataset. We verify the item, available countries, and time period, and confirm that Spain and Türkiye are both included.

In [6]:
# First, we check which item is included in the dataset
df["Item"].value_counts()

Item
Olives    2481
Name: count, dtype: int64

In [7]:
# Check how many countries are included
print("Number of countries:", df["Country"].nunique())

# Confirm that both countries used in our analysis are available
for country in ["Spain", "Türkiye"]:
    print(country, "->", country in df["Country"].values)

Number of countries: 62
Spain -> True
Türkiye -> True


In [8]:
# Check the available time period
print("First year:", df["Year"].min())
print("Last year:", df["Year"].max())
print("Number of years:", df["Year"].nunique())

First year: 1961
Last year: 2024
Number of years: 64


In [9]:
# Check the time coverage for Spain and Türkiye
target_countries = ["Spain", "Türkiye"]

coverage = (
    df[df["Country"].isin(target_countries)]
    .groupby("Country")["Year"]
    .agg(["min", "max", "count"])
)

coverage

,min,max,count
Country,,,
Spain,1961,2024,64
Türkiye,1961,2024,64


## 5. Data Quality Checks

Now, we focus on Spain and Türkiye and check for duplicates, missing values, and FAOSTAT flags before cleaning the data.

In [10]:
# Keep only the countries used in our analysis
target_df = df[df["Country"].isin(target_countries)].copy()

print("Rows:", target_df.shape[0])
target_df.head()

Rows: 128


,faostat,m49 code,Country,Item Code,Item,Year,Yield (kg/ha),Yield (kg/ha) flag,Production (tonnes),Production (tonnes) flag,Area harvested (ha),Area harvested (ha) flag
31,223,792,Türkiye,260,Olives,1961,1759.4,A,689324.0,A,391800.0,A
32,203,724,Spain,260,Olives,1961,NaN,NaN,1863400.0,A,NaN,M
37,203,724,Spain,260,Olives,1962,NaN,NaN,1641000.0,A,NaN,M
62,223,792,Türkiye,260,Olives,1962,726.6,A,290190.0,A,399393.0,A
77,223,792,Türkiye,260,Olives,1963,1528.9,A,618857.0,A,404760.0,A


In [11]:
# Check for duplicated country-year observations
duplicates = target_df.duplicated(subset=["Country", "Year"]).sum()

print("Duplicated country-year rows:", duplicates)

Duplicated country-year rows: 0


In [12]:
# Check missing values in the main variables
main_columns = [
    "Yield (kg/ha)",
    "Production (tonnes)",
    "Area harvested (ha)"
]

target_df[main_columns].isna().sum()

Yield (kg/ha)          19
Production (tonnes)     0
Area harvested (ha)    19
dtype: int64

In [13]:
# Check the FAOSTAT flags in the selected countries
flag_columns = [
    "Yield (kg/ha) flag",
    "Production (tonnes) flag",
    "Area harvested (ha) flag"
]

for column in flag_columns:
    print(f"\n{column}")
    print(target_df[column].value_counts(dropna=False))


Yield (kg/ha) flag
Yield (kg/ha) flag
A      107
NaN     19
E        2
Name: count, dtype: int64

Production (tonnes) flag
Production (tonnes) flag
A    128
Name: count, dtype: int64

Area harvested (ha) flag
Area harvested (ha) flag
A    107
M     19
E      2
Name: count, dtype: int64


### 5.1 Missing Value Coverage

Since yield and harvested area contain missing values, we check where they occur before deciding how to handle them.

In [14]:
# Check where the missing values occur
target_df[
    target_df["Yield (kg/ha)"].isna() |
    target_df["Area harvested (ha)"].isna()
][
    ["Country", "Year", "Yield (kg/ha)", "Area harvested (ha)"]
]

,Country,Year,Yield (kg/ha),Area harvested (ha)
32,Spain,1961,NaN,NaN
37,Spain,1962,NaN,NaN
96,Spain,1963,NaN,NaN
124,Spain,1964,NaN,NaN
154,Spain,1965,NaN,NaN
193,Spain,1966,NaN,NaN
212,Spain,1967,NaN,NaN
262,Spain,1968,NaN,NaN
265,Spain,1969,NaN,NaN
304,Spain,1970,NaN,NaN


All missing yield and harvested area values occur in Spain from 1961 to 1979, while production remains available for those years.

## 6. Data Cleaning

Now, we prepare the data for the next stages of the analysis. We keep the relevant variables, use simpler column names, and preserve the missing values found earlier.

In [15]:
# Keep only the variables needed for the analysis
clean_df = target_df[
    [
        "Country",
        "Year",
        "Production (tonnes)",
        "Production (tonnes) flag",
        "Area harvested (ha)",
        "Area harvested (ha) flag",
        "Yield (kg/ha)",
        "Yield (kg/ha) flag"
    ]
].copy()

# Rename the columns to make them easier to use
clean_df = clean_df.rename(columns={
    "Country": "country",
    "Year": "year",
    "Production (tonnes)": "production_tonnes",
    "Production (tonnes) flag": "production_flag",
    "Area harvested (ha)": "area_harvested_ha",
    "Area harvested (ha) flag": "area_harvested_flag",
    "Yield (kg/ha)": "yield_kg_ha",
    "Yield (kg/ha) flag": "yield_flag"
})

# Sort the data by country and year
clean_df = clean_df.sort_values(["country", "year"]).reset_index(drop=True)

clean_df.head()

,country,year,production_tonnes,production_flag,area_harvested_ha,area_harvested_flag,yield_kg_ha,yield_flag
0,Spain,1961,1863400.0,A,NaN,M,NaN,NaN
1,Spain,1962,1641000.0,A,NaN,M,NaN,NaN
2,Spain,1963,3124300.0,A,NaN,M,NaN,NaN
3,Spain,1964,572700.0,A,NaN,M,NaN,NaN
4,Spain,1965,1656100.0,A,NaN,M,NaN,NaN


## 7. Validate the Clean Data

Finally, we check that the cleaned dataset has the expected structure and that no observations were accidentally removed.

In [16]:
# Check the final shape
print("Rows:", clean_df.shape[0])
print("Columns:", clean_df.shape[1])

# Check for duplicated country-year observations
print(
    "Duplicated country-year rows:",
    clean_df.duplicated(subset=["country", "year"]).sum()
)

Rows: 128
Columns: 8
Duplicated country-year rows: 0


In [17]:
# Check the time coverage after cleaning
clean_df.groupby("country")["year"].agg(["min", "max", "count"])

,min,max,count
country,,,
Spain,1961,2024,64
Türkiye,1961,2024,64


In [18]:
# Confirm the remaining missing values
clean_df.isna().sum()

country                 0
year                    0
production_tonnes       0
production_flag         0
area_harvested_ha      19
area_harvested_flag     0
yield_kg_ha            19
yield_flag             19
dtype: int64

## 8. Save the Clean Data

Finally, we save the cleaned dataset so it can be used in the feature engineering and visualization stages.

In [19]:
# Save the cleaned dataset in the processed data folder
output_path = project_root / "data" / "processed" / "faostat_clean.csv"

clean_df.to_csv(output_path, index=False)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.


In [20]:
# Preview the final cleaned dataset
clean_df.head()

,country,year,production_tonnes,production_flag,area_harvested_ha,area_harvested_flag,yield_kg_ha,yield_flag
0,Spain,1961,1863400.0,A,NaN,M,NaN,NaN
1,Spain,1962,1641000.0,A,NaN,M,NaN,NaN
2,Spain,1963,3124300.0,A,NaN,M,NaN,NaN
3,Spain,1964,572700.0,A,NaN,M,NaN,NaN
4,Spain,1965,1656100.0,A,NaN,M,NaN,NaN
